In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
ROOT = "/kaggle/input/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy"
print(os.listdir(ROOT))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md



['dr_unified_v2', 'augmented_resized_V2']


In [2]:
for folder in ["dr_unified_v2", "augmented_resized_V2"]:
    print("=" * 60)
    print(folder)
    print("=" * 60)
    for dirpath, dirnames, filenames in os.walk(os.path.join(ROOT, folder)):
        if filenames:
            short = dirpath.replace(ROOT, "")
            others = [f for f in filenames if not f.lower().endswith((".jpg", ".jpeg", ".png"))]
            print(f"{short}  ->  {len(filenames)} files")
            print("   examples:", filenames[:5])
            if others:
                print("   NON-IMAGE files:", others[:10])

dr_unified_v2
/dr_unified_v2/dr_unified_v2/val/2  ->  1514 files
   examples: ['17327_right.jpg', '19785_left.jpg', '42765_left.jpg', '32665_left.jpg', '12890_left.jpg']
/dr_unified_v2/dr_unified_v2/val/0  ->  6895 files
   examples: ['34577_right.jpg', '28197_right.jpg', '40116_right.jpg', '7201_right.jpg', '3370_left.jpg']
/dr_unified_v2/dr_unified_v2/val/3  ->  125 files
   examples: ['166068a24416-GF.jpg', '42cc993f23a9-GF.jpg', '43223_right.jpg', '8a9bef2fbd4e-GF.jpg', '35124_right.jpg']
/dr_unified_v2/dr_unified_v2/val/1  ->  462 files
   examples: ['549_left.jpg', '6772_right.jpg', '7ccb267fd394.jpg', '40965_left.jpg', '6811_right.jpg']
/dr_unified_v2/dr_unified_v2/val/4  ->  249 files
   examples: ['35112_left.jpg', '6907_right.jpg', '15614_left.jpg', '40673_left.jpg', '31546_right.jpg']
/dr_unified_v2/dr_unified_v2/test/2  ->  1518 files
   examples: ['28938_left.jpg', '42297_right.jpg', '26676_left.jpg', '541db13517e2.jpg', '52dbec057cc8.jpg']
/dr_unified_v2/dr_unified_v2/tes

In [3]:
import re
from collections import Counter

base = os.path.join(ROOT, "dr_unified_v2", "dr_unified_v2")
counts = Counter()
other_examples = []

for split in ["train", "val", "test"]:
    for grade in ["0", "1", "2", "3", "4"]:
        for f in os.listdir(os.path.join(base, split, grade)):
            name = os.path.splitext(f)[0]
            if re.fullmatch(r"[0-9a-f]{12}-GF", name) or re.fullmatch(r"GF-[0-9a-f]{12}", name):
                counts["APTOS-GF"] += 1
            elif re.fullmatch(r"[0-9a-f]{12}", name):
                counts["APTOS"] += 1
            elif re.fullmatch(r"\d+_(left|right)", name):
                counts["EyePACS"] += 1
            else:
                counts["OTHER"] += 1
                if len(other_examples) < 20:
                    other_examples.append(f"{split}/{grade}/{f}")

print(counts)
print("Total:", sum(counts.values()))
print("\nOTHER examples:")
for e in other_examples:
    print("  ", e)

Counter({'EyePACS': 85178, 'APTOS-GF': 3662, 'APTOS': 3661})
Total: 92501

OTHER examples:


In [4]:
import re
import pandas as pd

base = os.path.join(ROOT, "dr_unified_v2", "dr_unified_v2")
rows = []
skipped_gf = 0

for split in ["train", "val", "test"]:
    for grade in ["0", "1", "2", "3", "4"]:
        folder = os.path.join(base, split, grade)
        for f in os.listdir(folder):
            name = os.path.splitext(f)[0]

            # 1. Skip Gaussian-filtered copies (both naming styles)
            if name.endswith("-GF") or name.startswith("GF-"):
                skipped_gf += 1
                continue

            # 2. Work out the source and the group (patient) ID
            eyepacs = re.fullmatch(r"(\d+)_(left|right)", name)
            if re.fullmatch(r"[0-9a-f]{12}", name):
                source, group_id = "APTOS", f"aptos_{name}"
            elif eyepacs:
                source, group_id = "EyePACS", f"eyepacs_{eyepacs.group(1)}"
            else:
                print("Unexpected file:", split, grade, f)
                continue

            # 3. Add one row for this image
            rows.append({
                "filename": f,
                "path": os.path.join(folder, f),
                "source": source,
                "group_id": group_id,
                "grade": int(grade),
                "label": int(int(grade) >= 1),   # 0 = No DR, 1 = DR (grades 1-4)
                "orig_split": split,
            })

df = pd.DataFrame(rows)

print("Skipped GF:", skipped_gf)
print("Rows:", len(df))
print("\nBy source:\n", df["source"].value_counts())
print("\nBy grade:\n", df["grade"].value_counts().sort_index())
print("\nDR %:", round(df["label"].mean() * 100, 1))
print("Duplicate filenames:", df["filename"].duplicated().sum())

leaky = (df.groupby("group_id")["orig_split"].nunique() > 1).sum()
print("Patients split across the ORIGINAL train/val/test:", leaky)

Skipped GF: 3662
Rows: 88839

By source:
 source
EyePACS    85178
APTOS       3661
Name: count, dtype: int64

By grade:
 grade
0    67148
1     4264
2    14152
3     1066
4     2209
Name: count, dtype: int64

DR %: 24.4
Duplicate filenames: 0
Patients split across the ORIGINAL train/val/test: 14244


In [5]:
from sklearn.model_selection import train_test_split

# One row per patient. A patient counts as DR if either eye has DR.
groups = df.groupby("group_id").agg(
    label=("label", "max"),
    source=("source", "first"),
).reset_index()
groups["strata"] = groups["source"] + "_" + groups["label"].astype(str)

# 70% train, 30% temp
train_g, temp_g = train_test_split(
    groups, test_size=0.30, stratify=groups["strata"], random_state=42
)
# Split temp into 15% val, 15% test
val_g, test_g = train_test_split(
    temp_g, test_size=0.50, stratify=temp_g["strata"], random_state=42
)

split_map = {}
split_map.update({g: "train" for g in train_g["group_id"]})
split_map.update({g: "val" for g in val_g["group_id"]})
split_map.update({g: "test" for g in test_g["group_id"]})
df["split"] = df["group_id"].map(split_map)

# Checks
print("Patients in more than one split:",
      (df.groupby("group_id")["split"].nunique() > 1).sum())
print("\nImages per split:\n", df["split"].value_counts())
print("\nDR % per split:\n",
      (df.groupby("split")["label"].mean() * 100).round(1))
print("\nSource mix per split:\n", pd.crosstab(df["split"], df["source"]))

Patients in more than one split: 0

Images per split:
 split
train    62195
val      13326
test     13318
Name: count, dtype: int64

DR % per split:
 split
test     24.4
train    24.4
val      24.7
Name: label, dtype: float64

Source mix per split:
 source  APTOS  EyePACS
split                 
test      548    12770
train    2563    59632
val       550    12776


In [6]:
OUT = "/kaggle/working"

for split in ["train", "val", "test"]:
    part = df[df["split"] == split].reset_index(drop=True)
    path = f"{OUT}/DR_{split}.csv"
    part.to_csv(path, index=False)
    print(f"DR_{split}.csv  ->  {len(part)} images,  DR %: {part['label'].mean()*100:.1f}")

DR_train.csv  ->  62195 images,  DR %: 24.4
DR_val.csv  ->  13326 images,  DR %: 24.7
DR_test.csv  ->  13318 images,  DR %: 24.4
